## connecting to external API to reconstruct the ShippingRegions table

In [ ]:
import pandas as pd

# create the Data Frame
shipping_regions_df = pd.read_csv('dataset/shipping_regions.csv')

# info of shipping_regions_df
shipping_regions_df.head()

# change Dtypes that are wrong
shipping_regions_df.region = shipping_regions_df['region'].astype(str)
shipping_regions_df.shipping_zone = shipping_regions_df['shipping_zone'].astype(str)
shipping_regions_df.estimated_days = shipping_regions_df['estimated_days'].astype('Int64')

# with this im gonna see if there is something that need to change
shipping_regions_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 24 entries, 0 to 23
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   country_code    24 non-null     str  
 1   region          0 non-null      str  
 2   shipping_zone   0 non-null      str  
 3   estimated_days  0 non-null      Int64
dtypes: Int64(1), str(3)
memory usage: 924.0 bytes


## Call to the API

Documents calls to investigate the documentation of an Public API, and consumn it. Later insert the the data or update the data  

In [ ]:
import requests

# request to endpoint of the API in cuestion
response = requests.get("https://api.first.org/data/v1/countries?limit=249")

# parse content to json
response_json = response.json()

# i needed the data of the response
data = response_json["data"]

# add regions to the df
for element in data:
    shipping_regions_df.loc[shipping_regions_df["country_code"] == element, 'region'] = data[element]['region']

# zone and day clasification by Zone convention
def classify_shipping(code):
    if code == 'PY':
        return 'Domestic', 'Zone 1', 2
    elif code in ['AR', 'BR', 'UY', 'BO']:
        return 'Mercosur Neighbors', 'Zone 2', 5
    elif code in ['CL', 'CO', 'PE']:
        return 'Rest of South America', 'Zone 3', 8
    elif code in ['US', 'CA', 'MX']:
        return 'North America', 'Zone 4', 12
    elif code in ['DE', 'ES', 'FR', 'GB', 'IT', 'NL', 'PT']:
        return 'Europe', 'Zone 5', 15
    else:
        return 'Rest of World', 'Zone 6', 20

# Applying the logic to the DataFrame
shipping_regions_df[['region', 'shipping_zone', 'estimated_days']] = shipping_regions_df.apply(
    lambda row: pd.Series(classify_shipping(row['country_code'])), axis=1
)

# Ensuring the data type is correct
shipping_regions_df['estimated_days'] = shipping_regions_df['estimated_days'].astype('Int64')



## Now add the new shipping reagios csv to the data base

In [27]:
from sqlite3 import connect
# create connection
conn = connect('data_base/e-commerse.db')
conn.execute('PRAGMA foreign_key = ON')

# insert the new csv to the db
shipping_regions_df.to_sql('ShippingRegions', conn, if_exists='replace', index=True)

# close the connection to db
conn.close()

## now the db is all set and done